In [1]:
# Cell 1 — Install/check dependencies
!pip install librosa soundfile tensorflow joblib pandas numpy scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Cell 2 — Imports
from pathlib import Path
import subprocess
import shutil
import json
import warnings

import numpy as np
import pandas as pd
import librosa
import joblib
import tensorflow as tf

warnings.filterwarnings("ignore")

In [3]:
# Cell 3 — Define local paths
CURRENT_DIR = Path.cwd()

PROJECT_ROOT = CURRENT_DIR.parents[2]
DEMO_ROOT = PROJECT_ROOT / "Demos" / "Real Life Trial Data (Demo 5)"

AUDIO_MODEL_ROOT = DEMO_ROOT / "train" / "audio_model"

PREPARED_AUDIO_DIR = AUDIO_MODEL_ROOT / "prepared_audio_data"
TRAINED_AUDIO_MODEL_DIR = AUDIO_MODEL_ROOT / "trained_audio_model"
AUDIO_OUTPUT_DIR = AUDIO_MODEL_ROOT / "audio_inference_outputs"
TEMP_AUDIO_DIR = AUDIO_MODEL_ROOT / "temp_inference_audio"

AUDIO_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

SCALER_PATH = PREPARED_AUDIO_DIR / "audio_feature_scaler.pkl"
FEATURE_INFO_PATH = PREPARED_AUDIO_DIR / "feature_info.json"

AUDIO_MODEL_PATH = TRAINED_AUDIO_MODEL_DIR / "final_audio_bilstm_model.keras"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Demo root:", DEMO_ROOT)
print("Prepared audio dir exists:", PREPARED_AUDIO_DIR.exists())
print("Trained audio model dir exists:", TRAINED_AUDIO_MODEL_DIR.exists())
print("Scaler exists:", SCALER_PATH.exists())
print("Feature info exists:", FEATURE_INFO_PATH.exists())
print("Audio model exists:", AUDIO_MODEL_PATH.exists())
print("Output dir:", AUDIO_OUTPUT_DIR)

Current dir: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train
Project root: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection
Demo root: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)
Prepared audio dir exists: True
Trained audio model dir exists: True
Scaler exists: True
Feature info exists: True
Audio model exists: True
Output dir: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\audio_inference_outputs


In [4]:
# Cell 4 — Check FFmpeg
ffmpeg_path = shutil.which("ffmpeg")

if ffmpeg_path is None:
    raise RuntimeError(
        "FFmpeg was not found in PATH. Install FFmpeg and restart VS Code/terminal."
    )

print("FFmpeg found at:", ffmpeg_path)

result = subprocess.run(
    ["ffmpeg", "-version"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

print(result.stdout.splitlines()[0])

FFmpeg found at: C:\Users\brian\AppData\Local\Microsoft\WinGet\Links\ffmpeg.EXE
ffmpeg version 8.1.1-full_build-www.gyan.dev Copyright (c) 2000-2026 the FFmpeg developers


In [5]:
# Cell 5 — Load config, scaler, and model
with open(FEATURE_INFO_PATH, "r") as file:
    feature_info = json.load(file)

scaler = joblib.load(SCALER_PATH)

audio_model = tf.keras.models.load_model(AUDIO_MODEL_PATH)

print("Feature info:")
print(json.dumps(feature_info, indent=4))

print("Loaded scaler:", type(scaler))
print("Loaded audio model:", AUDIO_MODEL_PATH.name)
audio_model.summary()

Feature info:
{
    "sample_rate": 16000,
    "window_seconds": 5.0,
    "stride_seconds": 2.5,
    "frame_length_seconds": 0.064,
    "frame_hop_seconds": 0.05,
    "target_audio_frames": 100,
    "n_mfcc": 13,
    "feature_count": 46,
    "features": [
        "mfcc_1_to_13",
        "delta_mfcc_1_to_13",
        "delta2_mfcc_1_to_13",
        "rms",
        "zero_crossing_rate",
        "spectral_centroid",
        "spectral_bandwidth",
        "spectral_rolloff",
        "pitch_f0",
        "voicing_probability"
    ],
    "label_mapping": {
        "truthful": 0,
        "deceptive": 1
    }
}
Loaded scaler: <class 'sklearn.preprocessing._data.StandardScaler'>
Loaded audio model: final_audio_bilstm_model.keras


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 100, 46)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, 100, 46)   │          0 │ input_layer[0][0] │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ masking (Masking)   │ (None, 100, 46)   │          0 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ any (Any)           │ (None, 100)       │          0 │ not_equal[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 100, 128)  │     56,832 │ masking[0][0],    │
│ (Bidirectional)     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 64)        │     41,216 │ bidirectional[0]… │
│ (Bidirectional)     │                   │            │ any[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ bidirectional_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 64)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 32)        │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         33 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 312,965 (1.19 MB)

 Trainable params: 104,321 (407.50 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 208,644 (815.02 KB)

In [6]:
# Cell 6 — Set inference configuration
SAMPLE_RATE = feature_info["sample_rate"]
WINDOW_SECONDS = feature_info["window_seconds"]
STRIDE_SECONDS = feature_info["stride_seconds"]
FRAME_LENGTH_SECONDS = feature_info["frame_length_seconds"]
FRAME_HOP_SECONDS = feature_info["frame_hop_seconds"]
TARGET_AUDIO_FRAMES = feature_info["target_audio_frames"]
N_MFCC = feature_info["n_mfcc"]
FEATURE_COUNT = feature_info["feature_count"]

print("Sample rate:", SAMPLE_RATE)
print("Window seconds:", WINDOW_SECONDS)
print("Stride seconds:", STRIDE_SECONDS)
print("Target audio frames:", TARGET_AUDIO_FRAMES)
print("Feature count:", FEATURE_COUNT)

Sample rate: 16000
Window seconds: 5.0
Stride seconds: 2.5
Target audio frames: 100
Feature count: 46


In [7]:
# Cell 7 — Choose test video
VIDEO_PATH = DEMO_ROOT / "Dataset" / "Real-life_Deception_Detection_2016" / "Clips" / "Deceptive" / "trial_lie_001.mp4"

print("Video path:", VIDEO_PATH)
print("Exists:", VIDEO_PATH.exists())

Video path: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\Dataset\Real-life_Deception_Detection_2016\Clips\Deceptive\trial_lie_001.mp4
Exists: True


In [8]:
# Cell 8 — Extract audio from video
def extract_audio_for_inference(video_path, output_dir):
    video_path = Path(video_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_wav_path = output_dir / f"{video_path.stem}_audio.wav"

    command = [
        "ffmpeg",
        "-y",
        "-i",
        str(video_path),
        "-vn",
        "-ac",
        "1",
        "-ar",
        str(SAMPLE_RATE),
        str(output_wav_path),
    ]

    result = subprocess.run(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"FFmpeg failed while extracting audio.\n{result.stderr}"
        )

    if not output_wav_path.exists():
        raise RuntimeError("Audio extraction failed. WAV file was not created.")

    return output_wav_path

In [14]:
# Test
wav_path = extract_audio_for_inference(VIDEO_PATH, TEMP_AUDIO_DIR)

print("Extracted audio:", wav_path)
print("Exists:", wav_path.exists())

Extracted audio: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\temp_inference_audio\trial_lie_001_audio.wav
Exists: True


In [9]:
# Cell 9 — Feature extraction helpers
def pad_or_trim_frames(feature_matrix, target_frames):
    current_frames = feature_matrix.shape[0]

    if current_frames == target_frames:
        return feature_matrix

    if current_frames > target_frames:
        return feature_matrix[:target_frames]

    pad_amount = target_frames - current_frames
    padding = np.zeros((pad_amount, feature_matrix.shape[1]), dtype=np.float32)

    return np.vstack([feature_matrix, padding])


def extract_audio_features_from_window(y_window, sr):
    frame_length = int(FRAME_LENGTH_SECONDS * sr)
    hop_length = int(FRAME_HOP_SECONDS * sr)

    if len(y_window) < frame_length:
        y_window = np.pad(y_window, (0, frame_length - len(y_window)))

    mfcc = librosa.feature.mfcc(
        y=y_window,
        sr=sr,
        n_mfcc=N_MFCC,
        n_fft=1024,
        hop_length=hop_length
    )

    delta_mfcc = librosa.feature.delta(mfcc)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)

    rms = librosa.feature.rms(
        y=y_window,
        frame_length=frame_length,
        hop_length=hop_length
    )

    zcr = librosa.feature.zero_crossing_rate(
        y_window,
        frame_length=frame_length,
        hop_length=hop_length
    )

    spectral_centroid = librosa.feature.spectral_centroid(
        y=y_window,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length
    )

    spectral_bandwidth = librosa.feature.spectral_bandwidth(
        y=y_window,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length
    )

    spectral_rolloff = librosa.feature.spectral_rolloff(
        y=y_window,
        sr=sr,
        n_fft=1024,
        hop_length=hop_length
    )

    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(
            y_window,
            fmin=librosa.note_to_hz("C2"),
            fmax=librosa.note_to_hz("C7"),
            sr=sr,
            frame_length=1024,
            hop_length=hop_length
        )

        f0 = np.nan_to_num(f0, nan=0.0)
        voiced_probs = np.nan_to_num(voiced_probs, nan=0.0)

    except Exception:
        frame_count = mfcc.shape[1]
        f0 = np.zeros(frame_count)
        voiced_probs = np.zeros(frame_count)

    min_frames = min(
        mfcc.shape[1],
        delta_mfcc.shape[1],
        delta2_mfcc.shape[1],
        rms.shape[1],
        zcr.shape[1],
        spectral_centroid.shape[1],
        spectral_bandwidth.shape[1],
        spectral_rolloff.shape[1],
        len(f0),
        len(voiced_probs)
    )

    feature_stack = np.vstack([
        mfcc[:, :min_frames],
        delta_mfcc[:, :min_frames],
        delta2_mfcc[:, :min_frames],
        rms[:, :min_frames],
        zcr[:, :min_frames],
        spectral_centroid[:, :min_frames],
        spectral_bandwidth[:, :min_frames],
        spectral_rolloff[:, :min_frames],
        f0[:min_frames].reshape(1, -1),
        voiced_probs[:min_frames].reshape(1, -1),
    ])

    feature_matrix = feature_stack.T.astype(np.float32)

    feature_matrix = pad_or_trim_frames(
        feature_matrix,
        TARGET_AUDIO_FRAMES
    )

    return feature_matrix

In [15]:
# Cell 10 — Create inference windows
def create_audio_inference_windows(wav_path):
    y, sr = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)

    duration = len(y) / sr

    window_samples = int(WINDOW_SECONDS * sr)
    stride_samples = int(STRIDE_SECONDS * sr)

    windows = []
    window_info = []

    if len(y) == 0:
        raise ValueError("Audio file is empty.")

    if len(y) < window_samples:
        padded_y = np.pad(y, (0, window_samples - len(y)))

        feature_matrix = extract_audio_features_from_window(padded_y, sr)

        windows.append(feature_matrix)
        window_info.append({
            "window_index": 0,
            "start": 0.0,
            "end": min(WINDOW_SECONDS, duration),
            "duration": duration,
        })

    else:
        start_sample = 0
        window_index = 0

        while start_sample + window_samples <= len(y):
            end_sample = start_sample + window_samples

            y_window = y[start_sample:end_sample]

            feature_matrix = extract_audio_features_from_window(y_window, sr)

            start_time = start_sample / sr
            end_time = end_sample / sr

            windows.append(feature_matrix)
            window_info.append({
                "window_index": window_index,
                "start": float(start_time),
                "end": float(end_time),
                "duration": float(duration),
            })

            start_sample += stride_samples
            window_index += 1

    X_infer = np.array(windows, dtype=np.float32)
    window_info_df = pd.DataFrame(window_info)

    return X_infer, window_info_df, duration

In [17]:
# Test
X_infer_raw, window_info_df, audio_duration = create_audio_inference_windows(wav_path)

print("X_infer_raw shape:", X_infer_raw.shape)
print("Audio duration:", audio_duration)
window_info_df.head()

X_infer_raw shape: (5, 100, 46)
Audio duration: 16.96


,window_index,start,end,duration
0,0,0.0,5.0,16.96
1,1,2.5,7.5,16.96
2,2,5.0,10.0,16.96
3,3,7.5,12.5,16.96
4,4,10.0,15.0,16.96


In [18]:
# Cell 11 — Scale inference features
def scale_audio_features_for_inference(X_raw, scaler):
    num_windows, time_steps, feature_count = X_raw.shape

    if feature_count != FEATURE_COUNT:
        raise ValueError(
            f"Feature count mismatch. Expected {FEATURE_COUNT}, got {feature_count}."
        )

    X_reshaped = X_raw.reshape(-1, feature_count)
    X_scaled_reshaped = scaler.transform(X_reshaped)

    X_scaled = X_scaled_reshaped.reshape(
        num_windows,
        time_steps,
        feature_count
    ).astype(np.float32)

    return X_scaled

In [19]:
# Test
X_infer = scale_audio_features_for_inference(X_infer_raw, scaler)

print("Scaled inference shape:", X_infer.shape)
print("NaN count:", np.isnan(X_infer).sum())
print("Inf count:", np.isinf(X_infer).sum())

Scaled inference shape: (5, 100, 46)
NaN count: 0
Inf count: 0


In [20]:
# Cell 12 — Predict audio scores
raw_scores = audio_model.predict(X_infer).ravel()

print("Raw scores shape:", raw_scores.shape)
print("First 10 scores:", raw_scores[:10])
print("Min:", raw_scores.min())
print("Max:", raw_scores.max())
print("Mean:", raw_scores.mean())

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 673ms/step
Raw scores shape: (5,)
First 10 scores: [0.0491608  0.05110551 0.05995097 0.06687169 0.04552944]
Min: 0.045529436
Max: 0.06687169
Mean: 0.054523684


In [21]:
# Cell 13 — Smooth audio predictions
def smooth_scores(scores, smoothing_window=3):
    scores = np.array(scores, dtype=np.float32)

    if len(scores) < smoothing_window:
        return scores

    smoothed = []

    half_window = smoothing_window // 2

    for i in range(len(scores)):
        start = max(0, i - half_window)
        end = min(len(scores), i + half_window + 1)

        smoothed.append(np.mean(scores[start:end]))

    return np.array(smoothed, dtype=np.float32)

In [26]:

smoothed_scores = smooth_scores(raw_scores, smoothing_window=3)

print("First 10 raw:", raw_scores[:10])
print("First 10 smoothed:", smoothed_scores[:10])

First 10 raw: [0.0491608  0.05110551 0.05995097 0.06687169 0.04552944]
First 10 smoothed: [0.05013315 0.05340576 0.05930939 0.0574507  0.05620056]


In [28]:
# Cell 14 — Risk labeling
def classify_audio_risk(score):
    if score >= 0.70:
        return "high"
    if score >= 0.45:
        return "medium"
    return "low"

In [29]:
# Cell 14 — Create Timeline
timeline_df = window_info_df.copy()

timeline_df["raw_score"] = raw_scores
timeline_df["smoothed_score"] = smoothed_scores
timeline_df["risk"] = timeline_df["smoothed_score"].apply(classify_audio_risk)

timeline_df["modality"] = "audio"

timeline_df.head()

,window_index,start,end,duration,raw_score,smoothed_score,risk,modality
0,0,0.0,5.0,16.96,0.049161,0.050133,low,audio
1,1,2.5,7.5,16.96,0.051106,0.053406,low,audio
2,2,5.0,10.0,16.96,0.059951,0.059309,low,audio
3,3,7.5,12.5,16.96,0.066872,0.057451,low,audio
4,4,10.0,15.0,16.96,0.045529,0.056201,low,audio


In [30]:
# Cell 15 — Add simple audio XAI explanation
def summarize_audio_window_features(feature_matrix):
    """
    feature_matrix is scaled, shape: (time_steps, feature_count)

    Because the features are scaled, we focus on variation patterns,
    not raw acoustic units.
    """

    feature_std = np.std(feature_matrix, axis=0)
    feature_mean_abs = np.mean(np.abs(feature_matrix), axis=0)

    mfcc_variation = float(np.mean(feature_std[0:13]))
    delta_variation = float(np.mean(feature_std[13:26]))
    delta2_variation = float(np.mean(feature_std[26:39]))

    rms_activity = float(feature_mean_abs[39])
    zcr_activity = float(feature_mean_abs[40])
    spectral_activity = float(np.mean(feature_mean_abs[41:44]))
    pitch_activity = float(feature_mean_abs[44])
    voicing_activity = float(feature_mean_abs[45])

    return {
        "mfcc_variation": mfcc_variation,
        "delta_mfcc_variation": delta_variation,
        "delta2_mfcc_variation": delta2_variation,
        "rms_activity": rms_activity,
        "zcr_activity": zcr_activity,
        "spectral_activity": spectral_activity,
        "pitch_activity": pitch_activity,
        "voicing_activity": voicing_activity,
    }


def generate_audio_explanation(feature_summary, score):
    factors = []

    if feature_summary["pitch_activity"] > 0.8:
        factors.append("higher pitch activity")

    if feature_summary["rms_activity"] > 0.8:
        factors.append("increased vocal energy variation")

    if feature_summary["mfcc_variation"] > 0.8:
        factors.append("higher spectral envelope variation")

    if feature_summary["delta_mfcc_variation"] > 0.8:
        factors.append("rapid short-term acoustic changes")

    if feature_summary["voicing_activity"] < 0.2:
        factors.append("lower voiced speech activity or possible silence")

    if not factors:
        factors.append("moderate acoustic variation")

    if score >= 0.70:
        summary = "High audio inconsistency was detected in this segment."
    elif score >= 0.45:
        summary = "Moderate audio inconsistency was detected in this segment."
    else:
        summary = "Low audio inconsistency was detected in this segment."

    detail = summary + " Main contributing patterns include " + ", ".join(factors) + "."

    return {
        "summary": summary,
        "detail": detail,
        "main_factors": factors,
        "feature_summary": feature_summary,
    }

In [31]:
# Apply explanations:
explanations = []

for i in range(len(X_infer)):
    feature_summary = summarize_audio_window_features(X_infer[i])
    explanation = generate_audio_explanation(
        feature_summary,
        float(smoothed_scores[i])
    )
    explanations.append(explanation)

timeline_df["explanation"] = explanations

timeline_df[["window_index", "start", "end", "smoothed_score", "risk", "explanation"]].head()

,window_index,start,end,smoothed_score,risk,explanation
0,0,0.0,5.0,0.050133,low,{'summary': 'Low audio inconsistency was detec...
1,1,2.5,7.5,0.053406,low,{'summary': 'Low audio inconsistency was detec...
2,2,5.0,10.0,0.059309,low,{'summary': 'Low audio inconsistency was detec...
3,3,7.5,12.5,0.057451,low,{'summary': 'Low audio inconsistency was detec...
4,4,10.0,15.0,0.056201,low,{'summary': 'Low audio inconsistency was detec...


In [32]:
# Cell 16 — Build final result object
def compute_overall_audio_result(timeline_df):
    overall_score = float(timeline_df["smoothed_score"].mean())

    high_count = int((timeline_df["risk"] == "high").sum())
    medium_count = int((timeline_df["risk"] == "medium").sum())
    low_count = int((timeline_df["risk"] == "low").sum())

    if overall_score >= 0.70:
        overall_risk = "high"
    elif overall_score >= 0.45:
        overall_risk = "medium"
    else:
        overall_risk = "low"

    return {
        "overall_score": overall_score,
        "overall_risk": overall_risk,
        "risk_counts": {
            "high": high_count,
            "medium": medium_count,
            "low": low_count,
        }
    }

In [33]:
# Create result:
overall_audio = compute_overall_audio_result(timeline_df)

audio_result = {
    "video_name": VIDEO_PATH.name,
    "audio_file": str(wav_path),
    "overall_score": overall_audio["overall_score"],
    "overall_risk": overall_audio["overall_risk"],
    "risk_counts": overall_audio["risk_counts"],
    "metadata": {
        "sample_rate": SAMPLE_RATE,
        "duration_seconds": float(audio_duration),
        "window_seconds": WINDOW_SECONDS,
        "stride_seconds": STRIDE_SECONDS,
        "num_windows": int(len(timeline_df)),
        "model_path": str(AUDIO_MODEL_PATH),
        "feature_count": FEATURE_COUNT,
        "target_audio_frames": TARGET_AUDIO_FRAMES,
    },
    "timeline": timeline_df.to_dict(orient="records"),
}

print("Video:", audio_result["video_name"])
print("Overall score:", audio_result["overall_score"])
print("Overall risk:", audio_result["overall_risk"])
print("Risk counts:", audio_result["risk_counts"])
print("Timeline items:", len(audio_result["timeline"]))

Video: trial_lie_001.mp4
Overall score: 0.05529991537332535
Overall risk: low
Risk counts: {'high': 0, 'medium': 0, 'low': 5}
Timeline items: 5


In [34]:
# Cell 17 — Save timeline CSV and JSON
def save_audio_timeline_outputs(audio_result, output_dir):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    video_stem = Path(audio_result["video_name"]).stem

    csv_path = output_dir / f"{video_stem}_audio_timeline.csv"
    json_path = output_dir / f"{video_stem}_audio_timeline.json"

    timeline_export_df = pd.DataFrame(audio_result["timeline"]).copy()

    if "explanation" in timeline_export_df.columns:
        timeline_export_df["explanation_summary"] = timeline_export_df["explanation"].apply(
            lambda item: item.get("summary", "") if isinstance(item, dict) else ""
        )
        timeline_export_df["explanation_detail"] = timeline_export_df["explanation"].apply(
            lambda item: item.get("detail", "") if isinstance(item, dict) else ""
        )
        timeline_export_df = timeline_export_df.drop(columns=["explanation"])

    timeline_export_df.to_csv(csv_path, index=False)

    with open(json_path, "w") as file:
        json.dump(audio_result, file, indent=4)

    return csv_path, json_path

In [35]:
# Save:
csv_path, json_path = save_audio_timeline_outputs(audio_result, AUDIO_OUTPUT_DIR)

print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)

Saved CSV: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\audio_inference_outputs\trial_lie_001_audio_timeline.csv
Saved JSON: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\audio_inference_outputs\trial_lie_001_audio_timeline.json


In [36]:
# Cell 18 — Preview saved timeline
preview_df = pd.read_csv(csv_path)

print("Saved timeline shape:", preview_df.shape)
preview_df.head()

Saved timeline shape: (5, 10)


,window_index,start,end,duration,raw_score,smoothed_score,risk,modality,explanation_summary,explanation_detail
0,0,0.0,5.0,16.96,0.049161,0.050133,low,audio,Low audio inconsistency was detected in this s...,Low audio inconsistency was detected in this s...
1,1,2.5,7.5,16.96,0.051106,0.053406,low,audio,Low audio inconsistency was detected in this s...,Low audio inconsistency was detected in this s...
2,2,5.0,10.0,16.96,0.059951,0.059309,low,audio,Low audio inconsistency was detected in this s...,Low audio inconsistency was detected in this s...
3,3,7.5,12.5,16.96,0.066872,0.057451,low,audio,Low audio inconsistency was detected in this s...,Low audio inconsistency was detected in this s...
4,4,10.0,15.0,16.96,0.045529,0.056201,low,audio,Low audio inconsistency was detected in this s...,Low audio inconsistency was detected in this s...


In [38]:
# Cell 19 — Wrap everything into one function
def predict_audio_timeline(video_path):
    video_path = Path(video_path)

    if not video_path.exists():
        raise FileNotFoundError(f"Video not found: {video_path}")

    wav_path = extract_audio_for_inference(video_path, TEMP_AUDIO_DIR)

    X_raw, window_info_df, audio_duration = create_audio_inference_windows(wav_path)

    X_scaled = scale_audio_features_for_inference(X_raw, scaler)

    raw_scores = audio_model.predict(X_scaled).ravel()
    smoothed_scores = smooth_scores(raw_scores, smoothing_window=3)

    timeline_df = window_info_df.copy()
    timeline_df["raw_score"] = raw_scores
    timeline_df["smoothed_score"] = smoothed_scores
    timeline_df["risk"] = timeline_df["smoothed_score"].apply(classify_audio_risk)
    timeline_df["modality"] = "audio"

    explanations = []

    for i in range(len(X_scaled)):
        feature_summary = summarize_audio_window_features(X_scaled[i])
        explanation = generate_audio_explanation(
            feature_summary,
            float(smoothed_scores[i])
        )
        explanations.append(explanation)

    timeline_df["explanation"] = explanations

    overall_audio = compute_overall_audio_result(timeline_df)

    result = {
        "video_name": video_path.name,
        "audio_file": str(wav_path),
        "overall_score": overall_audio["overall_score"],
        "overall_risk": overall_audio["overall_risk"],
        "risk_counts": overall_audio["risk_counts"],
        "metadata": {
            "sample_rate": SAMPLE_RATE,
            "duration_seconds": float(audio_duration),
            "window_seconds": WINDOW_SECONDS,
            "stride_seconds": STRIDE_SECONDS,
            "num_windows": int(len(timeline_df)),
            "model_path": str(AUDIO_MODEL_PATH),
            "feature_count": FEATURE_COUNT,
            "target_audio_frames": TARGET_AUDIO_FRAMES,
        },
        "timeline": timeline_df.to_dict(orient="records"),
    }

    return result

In [39]:
# Test:
test_result = predict_audio_timeline(VIDEO_PATH)

print("Video:", test_result["video_name"])
print("Overall score:", test_result["overall_score"])
print("Overall risk:", test_result["overall_risk"])
print("Timeline items:", len(test_result["timeline"]))
print("First timeline item:")
test_result["timeline"][0]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Video: trial_lie_001.mp4
Overall score: 0.05529991537332535
Overall risk: low
Timeline items: 5
First timeline item:


{'window_index': 0,
 'start': 0.0,
 'end': 5.0,
 'duration': 16.96,
 'raw_score': 0.049160800874233246,
 'smoothed_score': 0.050133153796195984,
 'risk': 'low',
 'modality': 'audio',
 'explanation': {'summary': 'Low audio inconsistency was detected in this segment.',
  'detail': 'Low audio inconsistency was detected in this segment. Main contributing patterns include higher pitch activity, higher spectral envelope variation, rapid short-term acoustic changes.',
  'main_factors': ['higher pitch activity',
   'higher spectral envelope variation',
   'rapid short-term acoustic changes'],
  'feature_summary': {'mfcc_variation': 0.8078128099441528,
   'delta_mfcc_variation': 1.0186246633529663,
   'delta2_mfcc_variation': 1.009541630744934,
   'rms_activity': 0.38253074884414673,
   'zcr_activity': 0.6400269865989685,
   'spectral_activity': 0.7177247405052185,
   'pitch_activity': 0.8758781552314758,
   'voicing_activity': 0.7597677111625671}}}

In [40]:
# Cell 20 — Save result from function
csv_path, json_path = save_audio_timeline_outputs(test_result, AUDIO_OUTPUT_DIR)

print("Saved CSV:", csv_path)
print("Saved JSON:", json_path)

Saved CSV: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\audio_inference_outputs\trial_lie_001_audio_timeline.csv
Saved JSON: d:\NSBM\3rd Year\Final Year Project\Product Development\Github\Mutimodal-Deception-Dectection\Demos\Real Life Trial Data (Demo 5)\train\audio_model\audio_inference_outputs\trial_lie_001_audio_timeline.json
